# 📓 Notebook 1: What is a Write-Ahead Log?

**The big question:** *"How do databases not lose your data when the power dies?"*

Imagine a simple key-value store that lives in memory: a Python dict. It is fast, but the moment the process crashes everything is gone. If we save the dict to disk on every write, we are slow — and worse, we can crash *halfway* through saving.

The trick is the **Write-Ahead Log (WAL)**: before changing anything, append a single line to a log file describing what we are about to do. Only then do we mutate state. If we crash, on restart we *replay* the log to rebuild the state.

In this notebook we compare:

1. 🟥 **BAD** — pure in-memory store: fast but loses everything on crash.
2. 🟨 **BETTER** — rewrite the whole state to disk on every write: slow and unsafe mid-write.
3. 🟩 **BEST** — a tiny WAL: append-only, durable, recoverable.

## Learning objectives
- Feel why naïve "save the whole file" is a bad idea.
- Implement a minimal append-only log.
- Replay a log to rebuild state.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/write-ahead-log
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if it doesn't appear.

In [ ]:
import os, json, tempfile, shutil

WORKDIR = tempfile.mkdtemp(prefix="wal_lab_")
print("workdir:", WORKDIR)

## 🟥 Approach 1 (BAD): pure in-memory KV store

Fast as can be, but everything is gone if the process dies.

In [ ]:
class MemoryKV:
    def __init__(self):
        self.data = {}
    def put(self, k, v): self.data[k] = v
    def get(self, k):    return self.data.get(k)

kv = MemoryKV()
kv.put("user:1", "alice")
kv.put("user:2", "bob")
print(kv.get("user:1"))
# imagine the process crashes here -> everything lost

## 🟨 Approach 2 (BETTER but slow + unsafe): rewrite the whole file every write

This survives a clean shutdown but has two problems:

1. Every write rewrites the entire file → O(n) per write.
2. If we crash *while writing*, the file is half-written and the data is corrupt.

In [ ]:
class WholeFileKV:
    def __init__(self, path):
        self.path = path
        self.data = {}
        if os.path.exists(path):
            with open(path) as f:
                self.data = json.load(f)

    def put(self, k, v):
        self.data[k] = v
        with open(self.path, "w") as f:
            json.dump(self.data, f)        # 😬 not atomic — crash here = corrupt
            f.flush(); os.fsync(f.fileno())

    def get(self, k): return self.data.get(k)

path = os.path.join(WORKDIR, "kv.json")
kv = WholeFileKV(path)
kv.put("user:1", "alice")
kv.put("user:2", "bob")
print("on disk:", open(path).read())

## 🟩 Approach 3 (BEST): Write-Ahead Log

The WAL is just a text file we **append** to. Each line is one operation:

```
{"op": "put", "k": "user:1", "v": "alice"}
{"op": "put", "k": "user:2", "v": "bob"}
{"op": "del", "k": "user:1"}
```

Properties that come for free:

- **Append-only writes** → O(1) per write, friendly to disks.
- **Atomic per line** → a crash in the middle of a line just means we ignore that incomplete line on replay.
- **Recovery** = read the log start to finish and re-apply each operation.

In [ ]:
class WalKV:
    def __init__(self, log_path):
        self.log_path = log_path
        self.data = {}
        self._replay()                     # rebuild state on startup
        self._log = open(log_path, "a")    # then open in append mode

    def _replay(self):
        if not os.path.exists(self.log_path):
            return
        with open(self.log_path) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    # the last line might be torn from a crash mid-write — skip it
                    print("  ⚠️ ignoring torn log line:", line[:40])
                    continue
                if rec["op"] == "put":
                    self.data[rec["k"]] = rec["v"]
                elif rec["op"] == "del":
                    self.data.pop(rec["k"], None)

    def _append(self, rec):
        self._log.write(json.dumps(rec) + "\n")
        self._log.flush()
        os.fsync(self._log.fileno())       # force the OS to write to disk

    def put(self, k, v):
        self._append({"op": "put", "k": k, "v": v})
        self.data[k] = v

    def delete(self, k):
        self._append({"op": "del", "k": k})
        self.data.pop(k, None)

    def get(self, k): return self.data.get(k)

    def close(self): self._log.close()

log_path = os.path.join(WORKDIR, "wal.log")
kv = WalKV(log_path)
kv.put("user:1", "alice")
kv.put("user:2", "bob")
kv.delete("user:1")
kv.close()

print("log on disk:")
print(open(log_path).read())

In [ ]:
# Now simulate a restart by creating a new instance pointing at the same log.
kv2 = WalKV(log_path)
print("recovered:", kv2.data)
kv2.close()

## 🔬 Torn-write demo

Earlier we *claimed* the WAL is safe against a crash mid-line. Let's prove it. We manually append half a JSON line to the log — that's what the tail of the file would look like if the process died before finishing the last write — and then reopen the store.

In [ ]:
# Append a broken (half-written) line, as if the process died mid-write.
with open(log_path, "a") as f:
    f.write('{"op": "put", "k": "user:3", "v": "char')   # no closing brace, no newline

print("raw log:")
print(open(log_path).read())
print("---")

# Recovery should skip the torn line and still rebuild the valid state.
kv3 = WalKV(log_path)
print("recovered after torn write:", kv3.data)
kv3.close()

shutil.rmtree(WORKDIR)
print("cleaned up")

## ✅ Recap

The WAL pattern gives us **durability** (data survives crashes), **good write performance** (just appends), and **simple recovery** (just replay).

Real databases (Postgres, SQLite, RocksDB, Kafka...) all use a variant of this idea. They add **checkpoints** (periodic snapshots so you don't have to replay forever) and **log compaction** — but the core is exactly what you just built.